# 式の命名とインスタンスへの保存

JijModeling には特定の式に名前をつける機能があり、以下のような場面で効力を発揮します：

1. モデル構築時に繰り返し使われる式を一時変数に束縛し再利用する
2. 複雑な式に対し名前をつけて、数式出力を見やすくする
3. 目的関数の部分項など、求解後に値を確認したい決定変数を含む式の情報を OMMX に保存させ、自動的に評価させる

本節では、これらの用途を念頭に JijModeling で名前つきの式を定義する方法について説明します。

## {py:class}`~jijmodeling.NamedExpr` クラス

JijModeling では、名前つきの式を表すクラスとして {py:class}`~jijmodeling.NamedExpr` クラスが提供されています。
決定変数やプレースホルダーと同様に、{py:meth}`Problem.NamedExpr() <jijmodeling.Problem.NamedExpr>` メソッドを使って宣言することができます。
{py:meth}`Problem.NamedExpr() <jijmodeling.Problem.NamedExpr>` は以下の引数を取ります：

| 引数 | 型 | 説明 |
| :-- | :--: | :-- |
| `name` | `str` | 名前つき式の名前。Decorator API では省略可能。 |
| `definition` | 必須。{py:data}`~jijmodeling.ExpressionLike` | 名前つき式の定義。JijModeling の式オブジェクトや、Python の数値、文字列、タプル、リスト、辞書、NumPy 配列など、式に変換可能なオブジェクトを指定できます。 |
| `description` | `Optional[str]` | 省略可。名前つき式の説明。数式出力や OMMX に保存される式の説明に使用されます。 |
| `latex` | `Optional[str]` | 省略可。名前つき式の $\LaTeX$ 表現。数式出力時に使用されます。 |
| `save_in_ommx` | `bool` | 省略可（デフォルト：`False`）。`True` にすると、後述する条件を満たす場合、OMMX インスタンスに {py:class}`ommx.v1.NamedFunction` として保存されます。 |

例を見てみましょう。ナップサック問題において、アイテム数$N$をインスタンスデータとして与えるのではなく、各アイテムの重さを表すプレースホルダー配列 $w$ の長さから推論することを考えます。
まずは、{py:meth}`~jijmodeling.Problem.NamedExpr` を使わずに定式化すると以下のようになります：

In [1]:
import jijmodeling as jm


@jm.Problem.define("Knapsack (Unnamed)", sense=jm.ProblemSense.MAXIMIZE)
def knapsack_unnamed(problem: jm.DecoratedProblem):
    W = problem.Float(description="maximum weight capacity of the knapsack")
    w = problem.Float(ndim=1, description="weight of each item")
    N = w.len_at(0)
    v = problem.Float(shape=(N,), description="value of each item")
    x = problem.BinaryVar(
        shape=(N,), description="$x_i = 1$ if item i is put in the knapsack"
    )

    problem += jm.sum(v[i] * x[i] for i in N)
    problem += problem.Constraint("Weight", jm.sum(w[i] * x[i] for i in N) <= W)


knapsack_unnamed

Problem(name="Knapsack (Unnamed)", sense=MAXIMIZE, objective=sum(w.len_at(0).map(lambda (i: natural): v[i] * x[i])), constraints={Weight: [Constraint(name="Weight", sense=LESS_THAN_EQUAL, left=sum(w.len_at(0).map(lambda (i: natural): w[i] * x[i])), right=W, shape=Scalar(Float)),],})

期待通り $W, w, v$ のみの三つのインスタンスデータを与えればよいようになっていますが、$N$ の定義式 `len_at(w, 0)` が定義中で展開されてしまっており、特に総和の範囲などがみづらくなっています。
そこで、$N$ を {py:meth}`~jijmodeling.Problem.NamedExpr` を使って定義してみましょう：

In [2]:
@jm.Problem.define("Knapsack", sense=jm.ProblemSense.MAXIMIZE)
def knapsack(problem: jm.DecoratedProblem):
    W = problem.Float(description="maximum weight capacity of the knapsack")
    w = problem.Float(ndim=1, description="weight of each item")
    # N が NamedExpr に 包まれている！
    N = problem.NamedExpr(w.len_at(0), description="Length of w")
    v = problem.Float(shape=(N,), description="value of each item")
    x = problem.BinaryVar(
        shape=(N,), description="$x_i = 1$ if item i is put in the knapsack"
    )

    problem += jm.sum(v[i] * x[i] for i in N)
    problem += problem.Constraint("Weight", jm.sum(w[i] * x[i] for i in N) <= W)


knapsack

Problem(name="Knapsack", sense=MAXIMIZE, objective=sum(N.map(lambda (i: natural): v[i] * x[i])), constraints={Weight: [Constraint(name="Weight", sense=LESS_THAN_EQUAL, left=sum(N.map(lambda (i: natural): w[i] * x[i])), right=W, shape=Scalar(Float)),],})

末尾の `Named Expressions` 節に $N$ の定義式が現れ、残りの数式中でも$N$として表示されるようになりました。
また、問題に対する `NamedExpr` の一覧辞書を `problem.named_exprs` で確認することができます：

In [3]:
knapsack.named_exprs

{'N': NamedExpr(name="N", definition=w.len_at(0), description="Length of w")}

$N$ は JijModeling のモデル中では独立した変数として扱われますが、インスタンスへのコンパイル時には自動的に定義が展開され、NamedExpr を使わない場合と同値なインスタンスが得られます。

In [4]:
knapsack_instance_data = {
    "v": [10, 13, 18, 31, 7, 15],
    "w": [11, 15, 20, 35, 10, 33],
    "W": 47,
}

instance_named = knapsack.eval(knapsack_instance_data)
instance_unnamed = knapsack_unnamed.eval(knapsack_instance_data)

assert instance_named.objective.almost_equal(instance_unnamed.objective)
assert instance_named.constraints[0].function.almost_equal(
    instance_unnamed.constraints[0].function
)

## {py:class}`~jijmodeling.NamedExpr` の OMMX インタンスへの保存

:::{admonition} NamedFunction のサポートは OMMX 2.5.0 以降
:class: important

OMMX に以下の機能を利用するのに必要な機能が入ったのは OMMX 2.5.0 です。
従って、以下のワークフローを利用するためには OMMX 2.5.0 以降が必要になることに留意してください。
:::

上の例では {py:class}`~jijmodeling.NamedExpr` の定義式にはプレースホルダーしか現れませんでしたが、実際には任意の式に名前をつけることができ、特に決定変数が現れるような式も命名することができます。
また、上述の通り `save_in_ommx` を `True` に設定すると、特定の条件を満たす場合 OMMX インスタンスに {py:class}`ommx.v1.NamedFunction` として保存されます。
{py:class}`ommx.v1.NamedFunction` は OMMX における {py:class}`jijmodeling.NamedExpr` の対応物であり、特に OMMX の関数（{py:class}`ommx.v1.Function`）＝「決定変数に関する実数値の関数」（以下**スカラー式**と呼びます）に名前をつけて保存することができるものです。
また、求解後の {py:class}`ommx.v1.Solution` オブジェクトでは、決定変数の値に基づいて自動的に値が計算されるという特徴もあります。

以上を踏まえ、`save_in_ommx=True` により OMMX インスタンスに保存できる条件は以下のいずれかです：

1. スカラー式
2. スカラー式を成分に持つ配列または辞書である
   - この場合、式は添え字ごとに個別の NamedFunction として分解されて保存されます。

上記以外の場合に `save_in_ommx=True` が指定された場合、`NamedExpr` の宣言時に例外となります。

例を見てみましょう。ナップサック問題において、評価後に実際にナップサックに入れられたアイテムの総重量や、個別のアイテムの単位重さ当りの価値の寄与を知りたかったとします。

In [5]:
@jm.Problem.define("Knapsack", sense=jm.ProblemSense.MAXIMIZE)
def knapsack_weight(problem: jm.DecoratedProblem):
    W = problem.Float(description="maximum weight capacity of the knapsack")
    w = problem.Float(ndim=1, description="weight of each item")
    N = problem.NamedExpr(w.len_at(0), description="Length of w")
    v = problem.Float(shape=(N,), description="value of each item")
    x = problem.BinaryVar(
        shape=(N,), description="$x_i = 1$ if item i is put in the knapsack"
    )
    total_weight = problem.NamedExpr(
        jm.sum(w[i] * x[i] for i in N),
        description="Total weight of items in the knapsack",
        save_in_ommx=True,
    )
    problem.NamedExpr(
        "V",
        v / w * x,
        description="Contribution of each item to the value per unit weight",
        save_in_ommx=True,
    )

    problem += jm.sum(v[i] * x[i] for i in N)
    problem += problem.Constraint("Weight", total_weight <= W)


knapsack_weight

Problem(name="Knapsack", sense=MAXIMIZE, objective=sum(N.map(lambda (i: natural): v[i] * x[i])), constraints={Weight: [Constraint(name="Weight", sense=LESS_THAN_EQUAL, left=total_weight, right=W, shape=Scalar(Float)),],})

総重みの項を `total_weight`、個別のアイテムの寄与を `V` という名前つき式に束縛し、`save_in_ommx=True` として OMMX インスタンスに保存させています。
まずはコンパイラを作成し、インスタンスを生成してみましょう。

In [6]:
compiler = jm.Compiler.from_problem(knapsack_weight, knapsack_instance_data)
instance = compiler.eval_problem(knapsack_weight)

インスタンスに含まれる `NamedFunction` の一覧は {py:meth}`ommx.v1.Instance.named_functions` や {py:meth}`ommx.v1.Instance.named_functions_df` プロパティで確認することができます：

In [7]:
instance.named_functions_df

,type,function,used_ids,name,subscripts,description,parameters.subscripts
id,,,,,,,
0,Linear,Function(0.9090909090909091*x0),{0},V,[0],Contribution of each item to the value per uni...,[0]
1,Linear,Function(0.8666666666666667*x1),{1},V,[1],Contribution of each item to the value per uni...,[1]
2,Linear,Function(0.9*x2),{2},V,[2],Contribution of each item to the value per uni...,[2]
3,Linear,Function(0.8857142857142857*x3),{3},V,[3],Contribution of each item to the value per uni...,[3]
4,Linear,Function(0.7*x4),{4},V,[4],Contribution of each item to the value per uni...,[4]
5,Linear,Function(0.45454545454545453*x5),{5},V,[5],Contribution of each item to the value per uni...,[5]
6,Linear,Function(11*x0 + 15*x1 + 20*x2 + 35*x3 + 10*x4...,"{0, 1, 2, 3, 4, 5}",total_weight,[],Total weight of items in the knapsack,[]


個別の NamedExpr に対応する NamedFunction の情報を得たい場合、{py:meth}`Compiler.get_named_function_id_by_name() <jijmodeling.Compiler.get_named_function_id_by_name>` メソッドに `NamedExpr` の名前を与えると、`save_in_ommx=True`の場合は添え字（スカラー式の場合は `()` のみ）から `NamedFunction` の ID への辞書が得られます：

In [8]:
contrib_dict = compiler.get_named_function_id_by_name("V")
print(contrib_dict)
assert contrib_dict is not None
instance.get_named_function_by_id(contrib_dict[(0,)])

{(0,): 0, (1,): 1, (2,): 2, (3,): 3, (4,): 4, (5,): 5}


NamedFunction(id=0, function=0.9090909090909091*x0, name="V", subscripts=[0])

一方、 `N` のように `save_in_ommx=False` の場合は、`NamedFunction` として保存されないため、`get_named_function_id_by_name()` は `None` を返します：

In [9]:
assert compiler.get_named_function_id_by_name("N") is None

それでは、これを OpenJij で解き、解の中の `total_weight` の値を確認してみましょう：

In [10]:
from ommx_openjij_adapter import OMMXOpenJijSAAdapter

solution = OMMXOpenJijSAAdapter.solve(
    instance,
    num_reads=100,
    num_sweeps=10,
    uniform_penalty_weight=1.6,
)

solution.named_functions_df

,value,used_ids,name,subscripts,description,parameters.subscripts
id,,,,,,
0,0.000000,{0},V,[0],Contribution of each item to the value per uni...,[0]
1,0.000000,{1},V,[1],Contribution of each item to the value per uni...,[1]
2,0.000000,{2},V,[2],Contribution of each item to the value per uni...,[2]
3,0.885714,{3},V,[3],Contribution of each item to the value per uni...,[3]
4,0.700000,{4},V,[4],Contribution of each item to the value per uni...,[4]
5,0.000000,{5},V,[5],Contribution of each item to the value per uni...,[5]
6,45.000000,"{0, 1, 2, 3, 4, 5}",total_weight,[],Total weight of items in the knapsack,[]


こうして解に対応する総重量や個別のアイテムの寄与の値が得られました。
今回のように単純な場合であれば、`weight` constraint の `value` の値を `W` と比較することで総重量を確認することもできます。
こうした用途の他にも、目的関数の一部の項の評価値を確認したい場合など、今回のように `NamedExpr` を OMMX インスタンスに保存すると便利なケースがあります。